# PepAnno: анализ и отбор признаков

Исследуем широкий, но конечный набор признаков, вычисляемых из последовательности:
длину, аминокислотный состав, физико-химические свойства, концы пептида,
локальные окна, простые шаблоны и 400 дипептидов.

Выбор групп — только по 3-кратной CV внутри train. Отложенная validation
используется один раз для проверки. Официальный test не участвует.



In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    raise RuntimeError('Запустите из корня проекта или каталога notebooks')
sys.path.insert(0, str(ROOT))
from src.load_dataset import load_data
from src.features_extractor import ACIDS, H, KD, extract_features

raw_train, official_test = load_data('pepanno')
source = raw_train.drop_duplicates(subset='seq').reset_index(drop=True)
train, val = train_test_split(source, test_size=0.2, random_state=42,
                              stratify=source.y_func)
train, val = train.reset_index(drop=True), val.reset_index(drop=True)
print(f'train={len(train)}, validation={len(val)}; test пока не используем при отборе')



Используем локальный файл: /Users/daniel/AMP_Scanner/data/pepanno/pepanno_train.csv
Используем локальный файл: /Users/daniel/AMP_Scanner/data/pepanno/pepanno_test.csv
PepAnno: исключено 476 строк с конфликтующими метками
Train: (16298, 4)
Test: (3072, 4)

TRAIN
Class 0: n=8149, mean=47.3, median=37.0, min=11, max=190
Class 1: n=8149, mean=36.8, median=27.0, min=11, max=100

TEST
Class 0: n=1536, mean=77.1, median=79.0, min=28, max=100
Class 1: n=1536, mean=62.9, median=63.0, min=16, max=100
train=13038, validation=3260; test пока не используем при отборе


## Признаки

Заряд — приближение (K+R-D-E)/L без pH и концевых групп. Гидрофобный
момент предполагает идеальную спираль. Окно 7 для короткой цепи заменяется
всей цепью. Дипептиды нормированы на L-1.



In [2]:
GROUPS = {
    'length': ['seq_len', 'log_len'],
    'composition': [f'acid_{a}' for a in ACIDS],
    'physicochemical': ['charge_density', 'positive_frac', 'negative_frac',
        'hydrophobic_frac', 'mean_hydropathy', 'hydropathy_std',
        'hydrophobic_moment_helix', 'hydrophobic_moment_beta',
        'aromatic_frac', 'polar_frac'],
    'termini': ['n5_charge', 'c5_charge', 'n10_charge', 'c10_charge',
        'n5_hydrophobic', 'c5_hydrophobic', 'n10_hydrophobic', 'c10_hydrophobic',
        'first_half_charge', 'second_half_charge',
        'first_half_hydrophobic', 'second_half_hydrophobic'],
    'local_windows': ['window7_hydro_min', 'window7_hydro_max',
        'window7_charge_min', 'window7_charge_max',
        'window7_hydro_std', 'window7_charge_std'],
    'patterns': ['entropy', 'unique_residues', 'longest_same_run',
        'longest_hydrophobic_run', 'hydrophobic_transitions',
        'charge_transitions', 'gly_pro_frac', 'cys_frac'],
    'dipeptides': [f'di_{a}{b}' for a in ACIDS for b in ACIDS],
}

def longest_run(values):
    best = current = 0
    for value in values:
        current = current + 1 if value else 0
        best = max(best, current)
    return best

def extra_features(seq):
    length = len(seq)
    hydro = np.array([KD[a] for a in seq], dtype=float)
    charge = np.array([int(a in 'KR') - int(a in 'DE') for a in seq], dtype=float)
    is_hydro = np.array([a in H for a in seq], dtype=float)
    window = min(7, length)
    hydro_windows = np.convolve(hydro, np.ones(window) / window, mode='valid')
    charge_windows = np.convolve(charge, np.ones(window) / window, mode='valid')
    counts = np.array([seq.count(a) for a in ACIDS], dtype=float)
    proportions = counts[counts > 0] / length
    result = {
        'log_len': np.log1p(length),
        'hydropathy_std': hydro.std(),
        'hydrophobic_moment_beta': abs(np.dot(hydro, (-1.0) ** np.arange(length))) / length,
        'aromatic_frac': sum(a in 'FWY' for a in seq) / length,
        'polar_frac': sum(a in 'NQST' for a in seq) / length,
        'window7_hydro_min': hydro_windows.min(),
        'window7_hydro_max': hydro_windows.max(),
        'window7_charge_min': charge_windows.min(),
        'window7_charge_max': charge_windows.max(),
        'window7_hydro_std': hydro_windows.std(),
        'window7_charge_std': charge_windows.std(),
        'entropy': -np.sum(proportions * np.log2(proportions)),
        'unique_residues': len(proportions),
        'longest_same_run': max(longest_run(a == residue for a in seq) for residue in ACIDS) / length,
        'longest_hydrophobic_run': longest_run(is_hydro) / length,
        'hydrophobic_transitions': np.mean(is_hydro[1:] != is_hydro[:-1]) if length > 1 else 0,
        'charge_transitions': np.mean(charge[1:] != charge[:-1]) if length > 1 else 0,
        'gly_pro_frac': sum(a in 'GP' for a in seq) / length,
        'cys_frac': seq.count('C') / length,
    }
    halves = (seq[:length // 2], seq[length // 2:])
    for size in (5, 10):
        for end, piece in (('n', seq[:size]), ('c', seq[-size:])):
            result[f'{end}{size}_charge'] = (piece.count('K') + piece.count('R')
                - piece.count('D') - piece.count('E')) / len(piece)
            result[f'{end}{size}_hydrophobic'] = sum(a in H for a in piece) / len(piece)
    for name, piece in zip(('first_half', 'second_half'), halves):
        result[f'{name}_charge'] = (piece.count('K') + piece.count('R')
            - piece.count('D') - piece.count('E')) / len(piece)
        result[f'{name}_hydrophobic'] = sum(a in H for a in piece) / len(piece)
    return result

def make_features(frame):
    base = extract_features(frame, 'seq').reset_index(drop=True)
    extra = pd.DataFrame([extra_features(s) for s in frame.seq])
    extra = extra.drop(columns=base.columns.intersection(extra.columns))
    vocabulary = {a + b: i for i, (a, b) in enumerate(
        (a, b) for a in ACIDS for b in ACIDS)}
    vectorizer = CountVectorizer(analyzer='char', ngram_range=(2, 2),
                                 lowercase=False, vocabulary=vocabulary)
    counts = vectorizer.transform(frame.seq)
    denom = np.maximum(frame.seq.str.len().to_numpy() - 1, 1)
    dipeptides = pd.DataFrame(counts.multiply(1 / denom[:, None]).toarray(),
                              columns=GROUPS['dipeptides'])
    result = pd.concat([base, extra, dipeptides], axis=1)
    assert not result.columns.duplicated().any()
    assert np.isfinite(result.to_numpy()).all()
    return result

X_train, X_val = make_features(train), make_features(val)
y_train, y_val = train.y_func.to_numpy(), val.y_func.to_numpy()
assert X_train.columns.equals(X_val.columns)
print('Число признаков:', X_train.shape[1])
print(pd.Series({name: len(cols) for name, cols in GROUPS.items()}))



Число признаков: 463
length               2
composition         20
physicochemical     10
termini             12
local_windows        6
patterns             8
dipeptides         400
dtype: int64


## Одномерные связи с меткой

Это описательная таблица на train. Корреляция сама по себе не показывает,
что признак добавит пользу к уже имеющимся.



In [3]:
univariate = []
for column in X_train:
    x = X_train[column].to_numpy()
    if np.std(x) == 0:
        continue
    rho, _ = spearmanr(x, y_train)
    auc = roc_auc_score(y_train, x)
    univariate.append({'feature': column, 'spearman': rho,
                       'single_auc': max(auc, 1 - auc),
                       'direction': '+' if auc >= .5 else '-'})
univariate = pd.DataFrame(univariate).sort_values('single_auc', ascending=False)
print(univariate.head(20).round(3).to_string(index=False))



           feature  spearman  single_auc direction
    charge_density     0.462       0.767         +
second_half_charge     0.443       0.755         +
            charge     0.394       0.727         +
        c10_charge     0.396       0.726         +
     positive_frac     0.390       0.725         +
     negative_frac    -0.387       0.721         -
            acid_M    -0.393       0.718         -
        n10_charge     0.346       0.697         +
window7_charge_min     0.341       0.694         +
 first_half_charge     0.335       0.693         +
        polar_frac    -0.334       0.693         -
           entropy    -0.324       0.687         -
            acid_D    -0.328       0.683         -
         c5_charge     0.315       0.676         +
            acid_E    -0.307       0.672         -
            acid_K     0.288       0.666         +
   unique_residues    -0.285       0.664         -
window7_charge_max     0.290       0.663         +
         n5_charge     0.276   

## Отбор групп на train

Для сравнения используем стандартизацию и L2 логистическую регрессию.
Преобразования обучаются внутри CV. Выбираем группы жадно по ROC-AUC;
прирост меньше 0.003 считаем недостаточным. CV после отбора оптимистична,
поэтому отдельно проверяем выбор на validation.



In [4]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
def model():
    return make_pipeline(StandardScaler(),
                         LogisticRegression(C=0.1, solver='liblinear', max_iter=2000))

def columns_for(group_names):
    return list(dict.fromkeys(c for group in group_names for c in GROUPS[group]))

cache = {}
def cv_auc(group_names):
    key = tuple(sorted(group_names))
    if key not in cache:
        values = cross_val_score(model(), X_train[columns_for(key)], y_train,
                                 cv=cv, scoring='roc_auc', n_jobs=1)
        cache[key] = values
    return cache[key]

single_groups = pd.DataFrame([
    {'group': group, 'features': len(GROUPS[group]),
     'cv_auc_mean': cv_auc([group]).mean(),
     'cv_auc_sd': cv_auc([group]).std(ddof=1)}
    for group in GROUPS
]).sort_values('cv_auc_mean', ascending=False)
print(single_groups.round(4).to_string(index=False))
print('Все группы: CV ROC-AUC =', round(cv_auc(list(GROUPS)).mean(), 4))



          group  features  cv_auc_mean  cv_auc_sd
     dipeptides       400       0.8820     0.0060
    composition        20       0.8781     0.0074
physicochemical        10       0.8563     0.0045
       patterns         8       0.8324     0.0059
        termini        12       0.7866     0.0086
  local_windows         6       0.7665     0.0047
         length         2       0.5955     0.0143
Все группы: CV ROC-AUC = 0.8955


In [5]:
selected, remaining, selection_rows = [], list(GROUPS), []
best_score = 0.5
while remaining:
    score, winner = max((cv_auc(selected + [g]).mean(), g) for g in remaining)
    gain = score - best_score
    accepted = not selected or gain >= 0.003
    selection_rows.append({'candidate': winner, 'cv_auc': score,
                           'gain': gain, 'accepted': accepted})
    if not accepted:
        break
    selected.append(winner)
    remaining.remove(winner)
    best_score = score
print(pd.DataFrame(selection_rows).round(4).to_string(index=False))
print('Выбранные группы:', selected)
print('Число признаков:', len(columns_for(selected)))



  candidate  cv_auc   gain  accepted
 dipeptides  0.8820 0.3820      True
     length  0.8883 0.0063      True
composition  0.8918 0.0035      True
    termini  0.8931 0.0013     False
Выбранные группы: ['dipeptides', 'length', 'composition']
Число признаков: 422


## Отбор отдельных признаков

На выбранных группах подбираем силу L1-регуляризации по той же CV.
Из моделей с ROC-AUC в пределах 0.003 от максимума выбираем наиболее
разреженную. Этот отбор также относится только к train.



In [6]:
group_columns = columns_for(selected)
def sparse_model(c):
    return make_pipeline(StandardScaler(),
                         LogisticRegression(C=c, l1_ratio=1.0, solver='liblinear',
                                            max_iter=2000))

sparse_rows = []
for c in (0.001, 0.003, 0.01, 0.03, 0.1):
    scores = cross_val_score(sparse_model(c), X_train[group_columns], y_train,
                             cv=cv, scoring='roc_auc', n_jobs=1)
    fitted = sparse_model(c).fit(X_train[group_columns], y_train)
    retained = int(np.count_nonzero(fitted[-1].coef_[0]))
    sparse_rows.append({'C': c, 'cv_auc': scores.mean(), 'cv_sd': scores.std(ddof=1),
                        'nonzero': retained})
sparse_results = pd.DataFrame(sparse_rows)
eligible = sparse_results[sparse_results.cv_auc >= sparse_results.cv_auc.max() - 0.003]
chosen_c = float(eligible.sort_values(['nonzero', 'C']).iloc[0].C)
sparse_fit = sparse_model(chosen_c).fit(X_train[group_columns], y_train)
print(sparse_results.round(4).to_string(index=False))
print('Выбранное C:', chosen_c)

def top_k_model(k):
    return make_pipeline(
        StandardScaler(),
        SelectFromModel(LogisticRegression(C=chosen_c, l1_ratio=1.0,
                        solver='liblinear', max_iter=2000),
                        threshold=-np.inf, max_features=k),
        LogisticRegression(C=0.1, solver='liblinear', max_iter=2000))

top_k_rows = []
for k in (10, 20, 40, 80, 160):
    scores = cross_val_score(top_k_model(k), X_train[group_columns], y_train,
                             cv=cv, scoring='roc_auc', n_jobs=1)
    top_k_rows.append({'k': k, 'cv_auc': scores.mean(), 'cv_sd': scores.std(ddof=1)})
top_k_results = pd.DataFrame(top_k_rows)
eligible_k = top_k_results[top_k_results.cv_auc >= top_k_results.cv_auc.max() - 0.003]
chosen_k = int(eligible_k.sort_values('k').iloc[0].k)
top_k_fit = top_k_model(chosen_k).fit(X_train[group_columns], y_train)
selected_features = [str(name) for name in np.array(group_columns)[top_k_fit[1].get_support()]]
print(top_k_results.round(4).to_string(index=False))
print('Выбрано признаков:', len(selected_features), 'из', len(group_columns))
print('Выбранные признаки:', selected_features)



    C  cv_auc  cv_sd  nonzero
0.001  0.8260 0.0105        9
0.003  0.8754 0.0088       15
0.010  0.8850 0.0065       96
0.030  0.8918 0.0058      242
0.100  0.8931 0.0062      350
Выбранное C: 0.03
  k  cv_auc  cv_sd
 10  0.8744 0.0051
 20  0.8877 0.0069
 40  0.8892 0.0057
 80  0.8892 0.0063
160  0.8893 0.0067
Выбрано признаков: 20 из 422
Выбранные признаки: ['di_CC', 'di_CR', 'di_CV', 'di_EE', 'di_QY', 'di_SL', 'di_TC', 'seq_len', 'log_len', 'acid_C', 'acid_D', 'acid_E', 'acid_G', 'acid_H', 'acid_K', 'acid_L', 'acid_M', 'acid_R', 'acid_T', 'acid_Y']


## Проверка на validation

Один раз сравниваем выбранные группы и компактный список с текущими 32
признаками проекта и полным набором. Результат не используем для отбора.



In [7]:
base_columns = list(extract_features(train.head(1), 'seq').columns)
choices = {'existing_32': base_columns,
           'selected_groups': columns_for(selected),
           'selected_features': selected_features,
           'all_groups': columns_for(GROUPS)}
validation_rows = []
for name, cols in choices.items():
    estimator = model().fit(X_train[cols], y_train)
    score = estimator.predict_proba(X_val[cols])[:, 1]
    validation_rows.append({'feature_set': name, 'n_features': len(cols),
                            'val_roc_auc': roc_auc_score(y_val, score),
                            'val_average_precision': average_precision_score(y_val, score)})
validation_results = pd.DataFrame(validation_rows).sort_values('val_roc_auc', ascending=False)
print(validation_results.round(4).to_string(index=False))
print('Компактный перечень:', selected_features)
best_cv_name = max(('selected_groups', 'all_groups'),
                   key=lambda name: cv_auc(selected if name == 'selected_groups' else list(GROUPS)).mean())
print('Лучший из полных наборов по CV train:', best_cv_name)
print('Признаки этого набора:', len(choices[best_cv_name]))



      feature_set  n_features  val_roc_auc  val_average_precision
       all_groups         458       0.9081                 0.9034
  selected_groups         422       0.9044                 0.9010
selected_features          20       0.8893                 0.8797
      existing_32          32       0.8871                 0.8699
Компактный перечень: ['di_CC', 'di_CR', 'di_CV', 'di_EE', 'di_QY', 'di_SL', 'di_TC', 'seq_len', 'log_len', 'acid_C', 'acid_D', 'acid_E', 'acid_G', 'acid_H', 'acid_K', 'acid_L', 'acid_M', 'acid_R', 'acid_T', 'acid_Y']
Лучший из полных наборов по CV train: all_groups
Признаки этого набора: 458


## Как трактовать результат

«Лучшие» здесь означает лучший проверенный набор для этой линейной модели
по CV train. Компактный список из 20 признаков — компромисс по размеру;
при меньшей validation ROC-AUC его нельзя назвать лучшим по качеству.
Результат не переносится автоматически на Random Forest и трансформер.
Официальный test уже просматривался в проекте, поэтому новые решения после
его просмотра нельзя считать независимой проверкой.


## Официальный PepAnno test: последующая проверка

Группы и компактный список уже выбраны выше по train/validation. Теперь модели с одинаковыми параметрами обучаются заново на **всей официальной обучающей части** PepAnno и оцениваются на официальном test. Для сравнения также переобучается текущий набор из 32 признаков. Test не используется для подбора признаков, гиперпараметров или порога. Основные показатели — ROC-AUC и average precision; accuracy, recall, F1 и MCC рассчитаны при фиксированном пороге 0.5. Разбиение test по длинам показывает, где меняется качество.

Это последующий исследовательский анализ: test уже использовался ранее в проекте, поэтому его нельзя считать полностью независимой оценкой новых решений. После просмотра этих результатов не следует выбирать набор по test и затем выдавать его показатель за подтверждённый результат.


In [8]:
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, recall_score

X_full_train = make_features(source)
X_official_test = make_features(official_test)
y_full_train = source.y_func.to_numpy()
y_official_test = official_test.y_func.to_numpy()
assert X_full_train.columns.equals(X_official_test.columns)

official_choices = {
    'existing_32': base_columns,
    'selected_features_20': selected_features,
    'selected_groups_422': columns_for(selected),
    'all_groups_458': columns_for(GROUPS),
}
test_rows = []
length_rows = []
lengths = official_test.seq.str.len().to_numpy()
for name, columns in official_choices.items():
    fitted = model().fit(X_full_train[columns], y_full_train)
    probabilities = fitted.predict_proba(X_official_test[columns])[:, 1]
    predicted = (probabilities >= 0.5).astype(int)
    test_rows.append({
        'feature_set': name, 'n_features': len(columns),
        'roc_auc': roc_auc_score(y_official_test, probabilities),
        'average_precision': average_precision_score(y_official_test, probabilities),
        'accuracy_at_0.5': accuracy_score(y_official_test, predicted),
        'recall_at_0.5': recall_score(y_official_test, predicted),
        'f1_at_0.5': f1_score(y_official_test, predicted),
        'mcc_at_0.5': matthews_corrcoef(y_official_test, predicted),
    })
    for lower, upper in ((31, 50), (51, 70), (71, 100)):
        subset = (lengths >= lower) & (lengths <= upper)
        length_rows.append({
            'feature_set': name, 'length': f'{lower}-{upper}',
            'n': int(subset.sum()), 'positives': int(y_official_test[subset].sum()),
            'roc_auc': roc_auc_score(y_official_test[subset], probabilities[subset]),
            'recall_at_0.5': recall_score(y_official_test[subset], predicted[subset]),
        })
test_results = pd.DataFrame(test_rows)
length_results = pd.DataFrame(length_rows)
print('Официальный test, модели переобучены на всём train:')
print(test_results.round(4).to_string(index=False))
print('\nПо длинам:')
print(length_results.round(4).to_string(index=False))


Официальный test, модели переобучены на всём train:
         feature_set  n_features  roc_auc  average_precision  accuracy_at_0.5  recall_at_0.5  f1_at_0.5  mcc_at_0.5
         existing_32          32   0.6373             0.6444           0.6019         0.3789     0.4876      0.2277
selected_features_20          20   0.6567             0.6642           0.6061         0.3815     0.4920      0.2376
 selected_groups_422         422   0.6676             0.6866           0.6240         0.4069     0.5198      0.2754
      all_groups_458         458   0.6777             0.6916           0.6305         0.4147     0.5289      0.2894

По длинам:
         feature_set length    n  positives  roc_auc  recall_at_0.5
         existing_32  31-50  562        427   0.7209         0.5363
         existing_32  51-70  775        399   0.5896         0.3509
         existing_32 71-100 1634        610   0.5845         0.2869
selected_features_20  31-50  562        427   0.7068         0.5948
selected_feature

### Вывод по официальному test

Полный набор из 458 признаков, выбранный по CV train, получил ROC-AUC 0.6777 и AP 0.6916. При одинаковом обучении и пороге 0.5 текущие 32 признака получили ROC-AUC 0.6373 и AP 0.6444. Компактный список из 20 признаков дал ROC-AUC 0.6567. Наиболее трудной остаётся группа длиной 71–100: ROC-AUC полного набора 0.6122. Следовательно, признаки улучшили линейную базовую модель, но не устранили сдвиг train/test. Результат относится к этому официальному test, уже просмотренному в проекте; для подтверждения понадобится новый внешний набор.
